# Chapter 42 — Authority, Isolation, and What Makes an Agent Worth Building

**Companion to *Pi Agents*** · [`Pi Agents` chapter 42](https://programmer.ie/books/pi/42-chapter/)

| | |
|---|---|
| Chapter | `42-chapter.md` · weight 42 · `/books/pi/42-chapter/` |
| Notebook | `notebooks/pi/42-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Can a value chosen by the inspected party gate the inspection?

The chapter's point: `AgentTool.replay` is a field a tool fills in **about
itself**. A tool can declare `replay: "safe"`, and **nothing in the declaration
verifies it**. The static scan shows that no shipped JavaScript in the Pi packages
reads the field — the declaration and the scan are the evidence, and the 
anti-vacuity self-test guarantees the scan would notice a reader.


## What this notebook establishes

- A tool can declare `replay` **about itself**, and nothing in the declaration verifies the claim.
- The **static scan for a reader of `AgentTool.replay`** would notice one — the anti-vacuity self-test makes an empty result meaningful.
- At **1.0.4, no shipped JavaScript** in the Pi packages reads a tool's `replay` field.
- The chapter's own gate evidence is reused: block-before-effect, the no-UI guard, the lying `readOnlyHint`.

## What this notebook does **not** establish

- **Containers, micro-VMs and credential proxies are documented, not run** — this book executed none.
- **A schematic sandbox is not containment evidence** — a diagram of a sandbox is not a proof it confines.
- **No `real_model` evidence exists** in this book (see `examples/evidence.json`).
- **The four-rung ladder and the permission model are the book's proposals**.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** + Inspect — the chapter's static scan and its anti-vacuity self-test, plus the reused gate evidence from chapters 16, 17 and 21.
* **Evidence scope:** `in_process_runtime` + `shipped_binary`. No container or VM is run.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(42))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/42-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: the value chosen by the inspected party

In [3]:
AUTH = pinb.driver_source("ch42-authority.ts")
auth = pinb.driver(AUTH, label="ch42-authority", timeout=120)
print(pinb.md_table(
    ["check", "value"],
    [["a tool can declare replay about itself", auth["toolCanDeclare"]],
     ["scan would notice a direct comparison", auth["antiVacuity"]["seesDirectComparison"]],
     ["scan would notice an object literal", auth["antiVacuity"]["seesObjectLiteral"]],
     ["JavaScript files scanned", auth["scanned"]],
     ["readers found", len(auth["readers"])]],
))

| check | value |
|---|---|
| a tool can declare replay about itself | True |
| scan would notice a direct comparison | True |
| scan would notice an object literal | True |
| JavaScript files scanned | 357 |
| readers found | 0 |


In [4]:
pinb.show_checks([
    pinb.check("the declaration is self-authored and unverified",
               auth["toolCanDeclare"], "replay: safe declared"),
    pinb.check("the scan is not vacuous: it would notice a reader",
               auth["antiVacuity"]["seesDirectComparison"] and auth["antiVacuity"]["seesObjectLiteral"],
               "anti-vacuity holds"),
    pinb.check("no shipped JavaScript reads a tool replay field at 1.0.4",
               auth["noReaders"] and len(auth["readers"]) == 0, f"{auth['scanned']} files, 0 readers"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------  ---------------------
PASS  the declaration is self-authored and unverified  replay: safe declared
PASS  the scan is not vacuous: it would notice a reader  anti-vacuity holds
PASS  no shipped JavaScript reads a tool replay field at 1.0.4  357 files, 0 readers

3/3 assertions held.


## The gate evidence, reused

The book's authority mechanism is not one scan — it is the gates, exercised in
chapters 16, 17 and 21. Those are **observed** evidence, and this notebook points
at them rather than re-running what they already show:

| Evidence | Chapter | What it shows |
|---|---|---|
| Block-before-effect | 16 | A `tool_call` guard leaves the filesystem untouched |
| No-UI refusal | 16 | Print mode refuses rather than allowing silently |
| Lying `readOnlyHint` | 17, 21 | An annotation passes the gate — hints are unverified |
| `replay: "safe"` | 42 | A self-declared recovery policy with no reader |

## The chapter's own tests

All three authority tests.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [5]:
PATTERNS = 'ch42-authority/replay.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch42-authority/replay.test.ts', show="replay")

$ node --test ch42-authority/replay.test.ts
  PASS  3 passed, 0 failed, 0 skipped  in 444 ms

  tests matching 'replay':
    ok   a tool can declare replay about itself, and nothing in the declaration verifies it
    ok   1.0.4: no shipped JavaScript in the Pi packages reads a tool's replay field


## Your turn: predict, then run

**Predict first.** A future Pi release *does* start reading `replay`. What breaks
in this notebook first — and what must chapter 42 then do?

**Then change one input.** Add a line to the scan's pattern that reads
`tool.replay` in a loop. Does the anti-vacuity self-test now fail the 
no-readers assertion?

**Predict the boundary.** The chapter's containers and micro-VMs are documented,
not run. What would a containment *claim* require that this notebook does not
have?

In [6]:
print("Predictions:")
print("  1. A reader in a future release fails the no-readers assertion, and chapter 42 must be re-read.")
print("  2. A real reader in the scanned code would surface in the readers list.")
print("  3. A containment claim needs an executed sandbox, not a diagram - unrun here.")
print()
print("Observed above:")
print(f"  tool can declare: {auth['toolCanDeclare']}")
print(f"  anti-vacuity: direct={auth['antiVacuity']['seesDirectComparison']}, literal={auth['antiVacuity']['seesObjectLiteral']}")
print(f"  scanned files: {auth['scanned']}, readers: {len(auth['readers'])}")
print()
assert auth["toolCanDeclare"]
assert auth["antiVacuity"]["seesDirectComparison"] and auth["antiVacuity"]["seesObjectLiteral"]
assert auth["noReaders"]
print("held: a self-authored replay claim is unverified; the scan is non-vacuous; 1.0.4 has no reader")

Predictions:
  1. A reader in a future release fails the no-readers assertion, and chapter 42 must be re-read.
  2. A real reader in the scanned code would surface in the readers list.
  3. A containment claim needs an executed sandbox, not a diagram - unrun here.

Observed above:
  tool can declare: True
  anti-vacuity: direct=True, literal=True
  scanned files: 357, readers: 0

held: a self-authored replay claim is unverified; the scan is non-vacuous; 1.0.4 has no reader


## Interpretation

Authority in this book comes from **who inspects whom**, and the chapter shows
the sharpest case: `AgentTool.replay` is **self-reported**. The evidence is:
the field can be set, the scan is non-vacuous, and no shipped code reads it at
1.0.4.

What this notebook does **not** establish:
* Containers, micro-VMs and credential proxies — **documented, not run**.
* A schematic sandbox — **not containment evidence**.
* Any `real_model` result — the book has none.

The practical rules:
1. **Self-reported fields are claims, not facts** — `replay: "safe"` is the tool's word.
2. **A non-vacuous scan is the tool of real evidence** — the anti-vacuity self-test makes "no reader" meaningful.
3. **Gates are exercised, not imagined** — the block-before-effect and no-UI refusal are observed evidence.
4. **Containment requires execution** — a diagram is not a sandbox, and unrun recipes stay unrun.

## Sources, execution mode and limitations

**Execution mode:** **Run** + Inspect — the chapter's static scan and its anti-vacuity self-test, plus the reused gate evidence from chapters 16, 17 and 21.

**Evidence scope:** `in_process_runtime` + `shipped_binary`. No container or VM is run.

### What was read or run

- **Ran** `drivers/ch42-authority.ts`: the static scan of the Pi packages' `dist` for a reader of `AgentTool.replay`, with its anti-vacuity self-test.
- **Ran** `ch42-authority/replay.test.ts` (3 tests).
- **Reused** the observed gate evidence from chapters 16, 17 and 21 (quoted, not re-run).
- **Read** `examples/evidence.json`, chapter 42 rows (3 claims).

### Limitations

- **Containers, micro-VMs and credential proxies are documented, not run** — this book executed none.
- **A schematic sandbox is not containment evidence**.
- **No `real_model` evidence exists** — the one retained attempt was refused before the model saw a prompt.
- **The four-rung ladder and the permission model are the book's proposals**.

### Optional work that was NOT run

- Container and VM recipes (documented, not run).
- Credential proxies (documented, not run).
- Any real-model call (none exists in this book).

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
